# Data Cleaning and Preprocessing

## Importing Libraries

In [1]:
import warnings

warnings.filterwarnings('ignore')

In [2]:
import pandas as pd
import os
import re
import nltk
import string
import numpy as np
import keras
from huggingface_hub import login
from datasets import load_dataset
import seaborn as sns
import matplotlib.pyplot as plt
import dagshub
import mlflow
from dotenv import load_dotenv
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import confusion_matrix, roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Embedding,SpatialDropout1D
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping


## Download dataset from huggingface

In [3]:
"""# Huggingface_hub Login
login()"""

"""# Loading the dataset into a pandas dataframe
df = pd.DataFrame(load_dataset("Tobi-Bueck/customer-support-tickets")['train'])

# Saving the dataset
df.to_csv("datasets/Customer_Support_Tickets.csv")"""

df = load_dataset("Tobi-Bueck/customer-support-tickets")['train'].to_pandas()

## Data Lookup

In [4]:
# First 5 rows
df.head()

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51.0,Security,Outage,Disruption,Data Breach,None,None,None,None
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51.0,Account,Disruption,Outage,IT,Tech Support,None,None,None
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51.0,Product,Feature,Tech Support,None,None,None,None,None
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51.0,Billing,Payment,Account,Documentation,Feedback,None,None,None
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51.0,Product,Feature,Feedback,Tech Support,None,None,None,None


In [5]:
# Last 5 rows
df.tail()

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
61760,Assistance Needed for IFTTT Docker Integration,I am facing integration problems with IFTTT Do...,I would be happy to assist with the IFTTT Dock...,Problem,Technical Support,low,en,NaN,Integration,Disruption,Performance,IT,Tech Support,None,None,None
61761,Bitten um Unterstützung bei der Integration,"Sehr geehrte Kundenservice, ich möchte die Int...","Sehr geehrte [Name], vielen Dank für Ihren Kon...",Change,Technical Support,medium,de,NaN,Integration,Feature,Documentation,Tech Support,None,None,None,None
61762,None,"Hello Customer Support, I am inquiring about t...",We will send you detailed information on plans...,Request,Billing and Payments,low,en,NaN,Billing,Payment,Feature,Feedback,Sales,Lead,None,None
61763,Hilfe bei digitalen Strategie-Problemen,Die Qualität unserer digitalen Strategie-Bearb...,Um den digitalen Strategie-Impuls zu überprüfe...,Incident,Product Support,high,de,NaN,Feedback,Performance,IT,Tech Support,None,None,None,None
61764,Optimierung Ihrer Datenanalyse-Plattform erlei...,"Sehr geehrte Customer Support-Team, ich schrei...","Sehr geehrter <name>, wir antworten Ihnen auf ...",Change,Sales and Pre-Sales,medium,de,NaN,Product,Feature,Performance,Guidance,Documentation,None,None,None


In [6]:
df.describe(include='all')

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
count,56466,61763,48576,48587,61765,61765,61765,28587.000000,48587,48528,48356,43990,27636,13225,5968,2472
unique,46635,53364,40139,4,52,5,2,NaN,211,350,591,831,956,920,755,512
top,Problembeschreibung,Facing issues with the integration of the Smar...,Das Downtime-Fehler-Szenario wird genauer unte...,Incident,Technical Support,medium,de,NaN,Security,Performance,IT,Tech Support,Tech Support,Tech Support,Tech Support,Documentation
freq,3,2,2,19444,14186,23378,33504,NaN,9156,8698,9430,10004,6904,2606,512,166
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,278.382027,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,165.962935,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,52.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,400.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,400.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61765 entries, 0 to 61764
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   subject   56466 non-null  object 
 1   body      61763 non-null  object 
 2   answer    48576 non-null  object 
 3   type      48587 non-null  object 
 4   queue     61765 non-null  object 
 5   priority  61765 non-null  object 
 6   language  61765 non-null  object 
 7   version   28587 non-null  float64
 8   tag_1     48587 non-null  object 
 9   tag_2     48528 non-null  object 
 10  tag_3     48356 non-null  object 
 11  tag_4     43990 non-null  object 
 12  tag_5     27636 non-null  object 
 13  tag_6     13225 non-null  object 
 14  tag_7     5968 non-null   object 
 15  tag_8     2472 non-null   object 
dtypes: float64(1), object(15)
memory usage: 7.5+ MB


## Data Cleaning

### Missing values

In [8]:
# Calculate the percentage of missing values in each row
def per_missing_values(df):
    return df.isna().mean() * 100

per_missing_values(df)

subject      8.579292
body         0.003238
answer      21.353517
type        21.335708
queue        0.000000
priority     0.000000
language     0.000000
version     53.716506
tag_1       21.335708
tag_2       21.431231
tag_3       21.709706
tag_4       28.778434
tag_5       55.256213
tag_6       78.588197
tag_7       90.337570
tag_8       95.997733
dtype: float64

### Removing rows with missing body

In [9]:
df = df[~df['body'].isna()]

## Data Preprocessing

In [10]:
def clean_text(text): 
    '''Make text lowercase, remove text in square brackets,remove links,remove punctuation and remove words containing numbers.''' 
    text = text.lower() 
    text = text.replace("\\n", ' ') 
    """text = re.sub(r'\s+', ' ', text).strip() 
    text = re.sub('\[.*?\]', '', text) 
    text = re.sub('https?://\S+|www\.\S+', '', text) 
    text = re.sub('<.*?>+', '', text) 
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text) 
    text = re.sub('\w*\d\w*', '', text)""" 
    return text 

def text_preprocessing(text): 
    """ Cleaning and parsing the text. """ 
    tokenizer = nltk.tokenize.RegexpTokenizer(r'\w+') 
    nopunc = clean_text(text) 
    tokenized_text = tokenizer.tokenize(nopunc) 
    #remove_stopwords = [w for w in tokenized_text if w not in stopwords.words('english')] 
    combined_text = ' '.join(tokenized_text) 
    return combined_text

In [11]:
# Combining the subject and body columns into a single column for text preprocessing

df['clean_subject'] = df['subject'].apply(str).apply(lambda x: text_preprocessing(x))
df['clean_body'] = df['body'].apply(str).apply(lambda x: text_preprocessing(x))

df['clean_sub_body'] = df['clean_subject'] + ' ' + df['clean_body']

In [12]:
# Combining the tag columns into a single column, making sure to drop any NaN values

"""cols = ['tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8']

# Combine the tag columns into a single column, making sure to drop any NaN values
df['tags'] = df[cols].apply(lambda x: ' '.join(x.dropna().astype(str)), axis=1)

# Adding tags to body and subject for better context in the model
df['clean_sub_body'] = df['clean_sub_body'] + " " + df['tags']"""

'cols = [\'tag_1\', \'tag_2\', \'tag_3\', \'tag_4\', \'tag_5\', \'tag_6\', \'tag_7\', \'tag_8\']\n\n# Combine the tag columns into a single column, making sure to drop any NaN values\ndf[\'tags\'] = df[cols].apply(lambda x: \' \'.join(x.dropna().astype(str)), axis=1)\n\n# Adding tags to body and subject for better context in the model\ndf[\'clean_sub_body\'] = df[\'clean_sub_body\'] + " " + df[\'tags\']'

In [13]:
# Classifying the tickets into departments based on the queue column

conditions = [
    df["queue"].str.contains(
        r"technical|api|authentication|integration|engineering|support|Service Outages and Maintenance|IT & Technology/Security Operations|IT & Technology/Network Infrastructure|IT & Technology/Software Development",
        case=False,
        na=False
    ),
    df["queue"].str.contains(
        r"sales|business development|account executive|General Inquiry",
        case=False,
        na=False
    ),
    df["queue"].str.contains(
        r"billing|payment|invoice|refund|Returns and Exchanges",
        case=False,
        na=False
    ),
    df["queue"].str.contains(
        r"customer success|account management|customer care|Customer Service",
        case=False,
        na=False
    ),
    df["queue"].str.contains(
        r"legal|compliance|privacy|gdpr|Law & Government/Government Services",
        case=False,
        na=False
    ),
    df['queue'].str.contains(
        r"hr|human resources|recruitment|talent acquisition|Human Resources",
        case=False,
        na=False
    )
]

choices = [
    "Technical",
    "Sales",
    "Billing",
    "Customer Success",
    "Legal",
    "HR"
]

df["department"] = np.select(
    conditions,
    choices,
    default="Other"
)

In [14]:
(df['department'].value_counts())

department
Technical           32259
Other               10426
Customer Success     7420
Billing              7312
Sales                2522
HR                   1204
Legal                 620
Name: count, dtype: int64

In [15]:
# Removing duplicate rows
df.drop_duplicates(inplace=True)

In [16]:
# Saving the cleaned dataset
df.to_csv("datasets/Customer_Support_Tickets_Cleaned.csv", index=False)

In [ ]:
#df = df[df['department'] != 'Other']

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8,clean_subject,clean_body,clean_sub_body,department
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51.0,Security,Outage,Disruption,Data Breach,None,None,None,None,wesentlicher sicherheitsvorfall,sehr geehrtes support team ich möchte einen gr...,wesentlicher sicherheitsvorfall sehr geehrtes ...,Technical
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51.0,Account,Disruption,Outage,IT,Tech Support,None,None,None,account disruption,dear customer support team i am writing to rep...,account disruption dear customer support team ...,Technical
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51.0,Product,Feature,Tech Support,None,None,None,None,None,query about smart home system integration feat...,dear customer support team i hope this message...,query about smart home system integration feat...,Billing
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51.0,Billing,Payment,Account,Documentation,Feedback,None,None,None,inquiry regarding invoice details,dear customer support team i hope this message...,inquiry regarding invoice details dear custome...,Billing
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51.0,Product,Feature,Feedback,Tech Support,None,None,None,None,question about marketing agency software compa...,dear support team i hope this message reaches ...,question about marketing agency software compa...,Sales
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61760,Assistance Needed for IFTTT Docker Integration,I am facing integration problems with IFTTT Do...,I would be happy to assist with the IFTTT Dock...,Problem,Technical Support,low,en,NaN,Integration,Disruption,Performance,IT,Tech Support,None,None,None,assistance needed for ifttt docker integration,i am facing integration problems with ifttt do...,assistance needed for ifttt docker integration...,Technical
61761,Bitten um Unterstützung bei der Integration,"Sehr geehrte Kundenservice, ich möchte die Int...","Sehr geehrte [Name], vielen Dank für Ihren Kon...",Change,Technical Support,medium,de,NaN,Integration,Feature,Documentation,Tech Support,None,None,None,None,bitten um unterstützung bei der integration,sehr geehrte kundenservice ich möchte die inte...,bitten um unterstützung bei der integration se...,Technical
61762,None,"Hello Customer Support, I am inquiring about t...",We will send you detailed information on plans...,Request,Billing and Payments,low,en,NaN,Billing,Payment,Feature,Feedback,Sales,Lead,None,None,none,hello customer support i am inquiring about th...,none hello customer support i am inquiring abo...,Billing
61763,Hilfe bei digitalen Strategie-Problemen,Die Qualität unserer digitalen Strategie-Bearb...,Um den digitalen Strategie-Impuls zu überprüfe...,Incident,Product Support,high,de,NaN,Feedback,Performance,IT,Tech Support,None,None,None,None,hilfe bei digitalen strategie problemen,die qualität unserer digitalen strategie bearb...,hilfe bei digitalen strategie problemen die qu...,Technical


# Baseline Model

## Data Preparation

In [17]:
vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    min_df=3
)

X = vectorizer.fit_transform(df["clean_sub_body"])

le = LabelEncoder()

y = le.fit_transform(df["department"])

In [18]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Model Training and Evaluation

### Support Vector Machine

In [19]:
svm_model = LinearSVC(
    C=1.0,
    random_state=42,
    max_iter=5000
)

svm_tags = {
    "algorithm": "Linear SVM",
    "model_type": "LinearSVC"
}

### Logistic Regression

In [20]:
# Model
lr_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

lr_tags = {
    'max_iter': 1000,
    'class_weight': 'balanced'
}

### Naive Bayes

In [21]:
nb_model = MultinomialNB()

### Evaluation Plots

In [22]:
# ---------------------------------------------
# ROC Curve Function
# ---------------------------------------------

def create_roc_curve(model, model_name, X_test, y_test, labels):
    
    n_classes = len(labels)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)

    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(X_test)

    elif isinstance(model, tf.keras.Model):
        y_score = model.predict(X_test)

    else:
        raise ValueError(
            f"{type(model).__name__} does not support "
            "predict_proba or decision_function."
        )

    # ======================
    # Binary Classification
    # ======================
    if n_classes == 2:

        if y_score.ndim > 1:
            y_score_binary = y_score[:, 1]
        else:
            y_score_binary = y_score

        fpr, tpr, _ = roc_curve(
            y_test,
            y_score_binary
        )

        roc_auc = auc(fpr, tpr)

        fig, ax = plt.subplots(figsize=(7, 5))

        ax.plot(
            fpr,
            tpr,
            lw=2,
            label=f"AUC = {roc_auc:.3f}"
        )

        ax.plot(
            [0, 1],
            [0, 1],
            linestyle="--",
            label="Random Classifier"
        )

    # ======================
    # Multiclass Classification
    # ======================
    else:

        # One-hot encode y_test
        y_test_bin = label_binarize(
            y_test,
            classes=np.arange(n_classes)
        )

        roc_auc = roc_auc_score(
            y_test_bin,
            y_score,
            multi_class="ovr",
            average="weighted"
        )

        fig, ax = plt.subplots(figsize=(8, 6))

        for i in range(n_classes):

            fpr, tpr, _ = roc_curve(
                y_test_bin[:, i],
                y_score[:, i]
            )

            class_auc = auc(fpr, tpr)

            ax.plot(
                fpr,
                tpr,
                lw=1.5,
                label=f"{labels[i]} (AUC={class_auc:.3f})"
            )

        ax.plot(
            [0, 1],
            [0, 1],
            linestyle="--",
            label="Random Classifier"
        )

    # ======================
    # Common Plot Formatting
    # ======================
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC Curve — {model_name}")

    ax.legend(
        loc="lower right",
        fontsize=8
    )

    plt.tight_layout()

    # Save plot
    save_dir = os.path.join(
        "Images",
        "ROC_Curves"
    )

    os.makedirs(
        save_dir,
        exist_ok=True
    )

    save_path = os.path.join(
        save_dir,
        f"{model_name}.png"
    )

    fig.savefig(
        save_path,
        dpi=150
    )

    plt.close(fig)

    print(
        f"ROC Curve saved → {save_path} "
        f"(AUC: {roc_auc:.3f})"
    )

    return save_path, roc_auc

# ---------------------------------------------
# Confusion Matrix Function
# ---------------------------------------------

def create_confusion_matrix(model_name, y_true, y_pred, class_names, path="Images"):
    
    cm = confusion_matrix(y_true, y_pred)

    save_dir = os.path.join(path, "Confusion Matrix")
    os.makedirs(save_dir, exist_ok=True)

    safe_name = model_name.replace(" ", "_")
    save_path = os.path.join(save_dir, f"{safe_name}.png")

    plt.figure(figsize=(8, 8))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )

    plt.title(f"{model_name} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()

    print(f"Confusion Matrix saved → {save_path}")

    return save_path

### Training and Evaluation

In [23]:
models = [
    (
        "Linear SVM",
        svm_model,
        (X_train, y_train),
        (X_test, y_test),
        {}
    ),
    (
        "Logistic Regression",
        lr_model,
        (X_train, y_train),
        (X_test, y_test),
        lr_tags
    ),
    (
        "Multinomial Naive Bayes",
        nb_model,
        (X_train, y_train),
        (X_test, y_test),
        {}
    )
]

In [24]:
model_reports = []
model_roc_paths = []
model_roc_auc_scores = []
model_cm_path = []

for model_name, model, train_set, test_set, tags in models:

    # Unpack train and test sets
    X_train, y_train = train_set
    X_test, y_test = test_set

    # Model Training
    model.fit(X_train, y_train)

    print(f"{model_name} trained successfully!")

    # Predictions and Evaluation
    y_pred = model.predict(X_test)

    labels = le.classes_
    report = classification_report(y_test, y_pred, target_names=labels, output_dict=True)
    model_reports.append(report)

    # ROC Curve
    roc_path, roc_auc = create_roc_curve(model, model_name, X_test, y_test, labels)
    model_roc_paths.append(roc_path)
    model_roc_auc_scores.append(roc_auc)
    print(f"{model_name} ROC AUC saved successfully!")

    # Confusion Matrix
    cm_path = create_confusion_matrix(
        model_name=model_name,
        y_true=y_test,
        y_pred=y_pred,
        class_names=le.classes_
    )
    model_cm_path.append(cm_path)

    print(f"{model_name} Confusion Matrix saved successfully!")
    print("--------------------------------------------------------")

Linear SVM trained successfully!
ROC Curve saved → Images\ROC_Curves\Linear SVM.png (AUC: 0.938)
Linear SVM ROC AUC saved successfully!
Confusion Matrix saved → Images\Confusion Matrix\Linear_SVM.png
Linear SVM Confusion Matrix saved successfully!
--------------------------------------------------------
Logistic Regression trained successfully!
ROC Curve saved → Images\ROC_Curves\Logistic Regression.png (AUC: 0.917)
Logistic Regression ROC AUC saved successfully!
Confusion Matrix saved → Images\Confusion Matrix\Logistic_Regression.png
Logistic Regression Confusion Matrix saved successfully!
--------------------------------------------------------
Multinomial Naive Bayes trained successfully!
ROC Curve saved → Images\ROC_Curves\Multinomial Naive Bayes.png (AUC: 0.838)
Multinomial Naive Bayes ROC AUC saved successfully!
Confusion Matrix saved → Images\Confusion Matrix\Multinomial_Naive_Bayes.png
Multinomial Naive Bayes Confusion Matrix saved successfully!
--------------------------------

### Model Tracking in Dagshub

In [25]:
dagshub.init(repo_owner='JS-Tharun', repo_name='Customer-Support-Automation', mlflow=True)

load_dotenv()

os.environ['MLFLOW_TRACKING_USERNAME'] = f"{os.getenv('MLFLOW_USERNAME')}"
os.environ['MLFLOW_TRACKING_PASSWORD'] = f"{os.getenv('MLFLOW_PASSWORD')}"

mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
mlflow.set_experiment(os.environ["MLFLOW_EXPERIMENT_NAME"])

for i, element in enumerate(models):
    model_name = element[0]
    model = element[1]
    X_train, y_train = element[2]
    X_test, y_test = element[3]
    model_tags = element[4]

    model_report = model_reports[i]
    model_roc_auc = model_roc_auc_scores[i]
    roc_path = model_roc_paths[i]

    with mlflow.start_run(run_name=model_name):

        # Log model parameters
        mlflow.set_tags(tags)

        mlflow.sklearn.log_model(model, model_name)

        # Log Model performance metrics
        mlflow_metrics = {}
        for label, metrics in model_report.items():
            if isinstance(metrics, dict):  # class-wise or avg metrics
                for metric_name, value in metrics.items():
                    if metric_name != "support":  # optional: skip support (not a metric)
                        key = f"{label}_{metric_name}"
                        mlflow_metrics[key] = float(value)
            else:
                # accuracy case (single value)
                mlflow_metrics[label] = float(metrics)
        mlflow.log_metrics(mlflow_metrics)

        # Log ROC AUC metric
        mlflow_metrics["roc_auc"] = float(model_roc_auc)   # ← AUC as a metric
        mlflow.log_metrics(mlflow_metrics)
        print("Metrics Logged")

        # Log ROC AUC curve image
        mlflow.log_artifact(roc_path, artifact_path="plots/ROC AUC Curve")

        # Log Confusion matrix image
        mlflow.log_artifact(model_cm_path[i], artifact_path="plots/Confusion Matrix")


Accessing as JS-Tharun

Initialized MLflow to track repo "JS-Tharun/Customer-Support-Automation"

Repository JS-Tharun/Customer-Support-Automation initialized!

2026/06/03 13:45:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/03 13:45:40 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Metrics Logged
🏃 View run Linear SVM at: https://dagshub.com/JS-Tharun/Customer-Support-Automation.mlflow/#/experiments/0/runs/b04959d3477f473c99343f64883b4300
🧪 View experiment at: https://dagshub.com/JS-Tharun/Customer-Support-Automation.mlflow/#/experiments/0


2026/06/03 13:45:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/03 13:45:53 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Metrics Logged
🏃 View run Logistic Regression at: https://dagshub.com/JS-Tharun/Customer-Support-Automation.mlflow/#/experiments/0/runs/6dafc65c841e46ee8b3378e74626a6f0
🧪 View experiment at: https://dagshub.com/JS-Tharun/Customer-Support-Automation.mlflow/#/experiments/0


2026/06/03 13:46:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/03 13:46:09 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Metrics Logged
🏃 View run Multinomial Naive Bayes at: https://dagshub.com/JS-Tharun/Customer-Support-Automation.mlflow/#/experiments/0/runs/9cdde47b183a45d58c28eb5226df23c2
🧪 View experiment at: https://dagshub.com/JS-Tharun/Customer-Support-Automation.mlflow/#/experiments/0


# Neural Network Models

## Dataset Preparation

In [26]:

# Word Tokenization
MAX_WORDS = 20000
MAX_SEQ_LEN = 200
EMBEDDING_DIM = 100
tokenizer = Tokenizer(num_words=MAX_WORDS)
tokenizer.fit_on_texts(df['clean_sub_body'].values)
word_index = tokenizer.word_index
print('Found %s unique tokens.' % len(word_index))

Found 39461 unique tokens.


In [27]:
# Paddding the sequences to ensure uniform input length for the model
X = tokenizer.texts_to_sequences(df['clean_sub_body'].values)
X = pad_sequences(X, maxlen=MAX_SEQ_LEN)
print('Shape of data tensor:', X.shape)

le = LabelEncoder()

Y = le.fit_transform(df["department"])

print('Shape of label tensor:', Y)

Shape of data tensor: (61763, 200)
Shape of label tensor: [6 6 0 ... 0 6 5]


### Train Test Split

In [28]:
#Train test split
X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size = 0.10, random_state = 42)
print(X_train.shape,Y_train.shape)
print(X_test.shape,Y_test.shape)

(55586, 200) (55586,)
(6177, 200) (6177,)


## Model Pipeline

### LSTM Pipeline

In [29]:
# LSTM

lstm_tags = {
    "frameword": "tensorflow.keras",
    "model_type": "LSTM"
}

lstm_train_params = {
    'optimizer': 'adam',
    'loss': 'sparse_categorical_crossentropy',
    'metrics': ['accuracy'],
    'epochs': 30,
    'batch_size': 256
}

lstm_model_params = {
    "max_words": MAX_WORDS,
    "embedding_dim": EMBEDDING_DIM,
    "input_length": X.shape[1],
    "lstm_units": 100,
    "dropout": 0.2,
    #"recurrent_dropout": 0.2
}


lstm_model = Sequential([
    Embedding(
        lstm_model_params["max_words"], 
        lstm_model_params["embedding_dim"], 
        input_length=lstm_model_params["input_length"]
    ),
    SpatialDropout1D(
        lstm_model_params["dropout"]
    ),
    LSTM(
        lstm_model_params["lstm_units"], 
        dropout=lstm_model_params["dropout"], 
        #recurrent_dropout=lstm_model_params["recurrent_dropout"]
    ),
    Dense(
        len(le.classes_), 
        activation='softmax'
    )
])

## Model Training

In [30]:
models = [
    (
        "LSTM",
        lstm_model,
        (X_train, Y_train),
        (X_test, Y_test),
        lstm_model_params,
        lstm_train_params,
        lstm_tags
    )
]

In [31]:
model_reports = []
model_history = []
model_roc_paths = []
model_roc_auc_scores = []
model_cm_path = []

for model_name, model, train_set, test_set, model_params, train_params, tags in models:

    X_train, Y_train = train_set
    X_test, Y_test = test_set
    

    # Model Training
    model.compile(
        optimizer=train_params['optimizer'],
        loss=train_params['loss'],
        metrics=train_params['metrics']
    )


    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        min_delta=0.001
    )


    history = model.fit(
        X_train,
        Y_train,
        epochs=train_params['epochs'],
        batch_size=train_params['batch_size'],
        callbacks=[early_stop]
    )
    model_history.append(history)


    # Model Evaluation
    predictions = model.predict(X_test)
    y_pred = np.argmax(predictions, axis=1)


    # Saving Keras Model Locally
    model_save_dir = "saved_models"
    os.makedirs(model_save_dir, exist_ok=True)
    model_save_path = os.path.join(model_save_dir, f"{model_name}.keras")
    model.save(model_save_path)

    print(f"{model_name} saved to {model_save_path}")


    # Classification Report
    labels = le.classes_
    report = classification_report(
        Y_test,
        y_pred,
        target_names=labels,
        output_dict= True
    )
    model_reports.append(report)


    # ROC Curve
    roc_path, roc_auc = create_roc_curve(model, model_name, X_test, Y_test, labels)
    model_roc_paths.append(roc_path)
    model_roc_auc_scores.append(roc_auc)
    print(f"{model_name} ROC AUC saved successfully!")


    # Confusion Matrix
    cm_path = create_confusion_matrix(
        model_name=model_name,
        y_true=Y_test,
        y_pred=y_pred,
        class_names=le.classes_
    )
    model_cm_path.append(cm_path)
    print(f"{model_name} Confusion Matrix saved successfully!")
    print("--------------------------------------------------------")

Epoch 1/30
218/218 [==============================] - 13s 32ms/step - loss: 1.0969 - accuracy: 0.6407
Epoch 2/30
218/218 [==============================] - 7s 31ms/step - loss: 0.8433 - accuracy: 0.7207
Epoch 3/30
218/218 [==============================] - 7s 31ms/step - loss: 0.7580 - accuracy: 0.7425
Epoch 4/30
218/218 [==============================] - 7s 30ms/step - loss: 0.7065 - accuracy: 0.7624
Epoch 5/30
218/218 [==============================] - 7s 30ms/step - loss: 0.6223 - accuracy: 0.7873
Epoch 6/30
218/218 [==============================] - 6s 30ms/step - loss: 0.5713 - accuracy: 0.8047
Epoch 7/30
218/218 [==============================] - 6s 29ms/step - loss: 0.5230 - accuracy: 0.8205
Epoch 8/30
218/218 [==============================] - 6s 29ms/step - loss: 0.4889 - accuracy: 0.8336
Epoch 9/30
218/218 [==============================] - 6s 29ms/step - loss: 0.4461 - accuracy: 0.8474
Epoch 10/30
218/218 [==============================] - 6s 28ms/step - loss: 0.4080 - accur

In [32]:
dagshub.init(repo_owner='JS-Tharun', repo_name='Customer-Support-Automation', mlflow=True)

load_dotenv()

os.environ['MLFLOW_TRACKING_USERNAME'] = f"{os.getenv('MLFLOW_USERNAME')}"
os.environ['MLFLOW_TRACKING_PASSWORD'] = f"{os.getenv('MLFLOW_PASSWORD')}"

mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
mlflow.set_experiment(os.environ["MLFLOW_EXPERIMENT_NAME"])

for i, element in enumerate(models):
    model_name = element[0]
    model = element[1]
    X_train, Y_train = element[2]
    X_test, Y_test = element[3]
    model_params = element[4]
    train_params = element[5]
    tags = element[6]

    model_report = model_reports[i]
    history = model_history[i]
    model_roc_auc = model_roc_auc_scores[i]
    roc_path = model_roc_paths[i]

    with mlflow.start_run(run_name=model_name):

        # ---------------------------------------------------
        # Logging Parameters and Tags
        # ---------------------------------------------------

        mlflow.set_tags(tags)

        mlflow.log_params(model_params)
        mlflow.log_params(train_params)

        print(f"{model_name} parameters and tags logged!")

        #-----------------------------------------------------
        # Model training and validation performance tracking
        #-----------------------------------------------------

        for epoch in range(len(history.history['loss'])):
            mlflow.log_metric("train_loss", history.history['loss'][epoch], step=epoch)
            mlflow.log_metric("train_accuracy", history.history['accuracy'][epoch], step=epoch)

        print(f"{model_name} training and validation performance logged!")

        #------------------------------------------------------
        # Log Model
        #------------------------------------------------------

        model_path = os.path.join("saved_models", f"{model_name}.keras")

        if os.path.exists(model_path):
            keras_model = keras.models.load_model(model_path)
            model_info = mlflow.keras.log_model(
                keras_model,
                artifact_path=model_name
            )
            print(f"{model_name} model logged!")

        else:
            print(f"{model_name} model file not found at {model_path}!")


        #------------------------------------------------------
        # Model testing performance tracking
        #------------------------------------------------------

        # Log best epoch metrics explicitly (for clean experiment table comparison)
        best_epoch = np.argmin(history.history['loss'])
        mlflow.log_metric("best_epoch", best_epoch)
        mlflow.log_metric("best_train_accuracy", history.history['accuracy'][best_epoch])

        # Link test metrics to the LoggedModel using model_id
        mlflow_metrics = {}
        for label, metrics in model_report.items():
            if isinstance(metrics, dict):
                for metric_name, value in metrics.items():
                    if metric_name != "support":
                        mlflow_metrics[f"{label}_{metric_name}"] = float(value)
            else:
                mlflow_metrics[label] = float(metrics)

        # Pass model_id to link metrics to the model in the Models tab
        mlflow.log_metrics(mlflow_metrics, model_id=model_info.model_id)

        print(f"{model_name} metrics logged")

        # Log ROC AUC metric
        mlflow_metrics["roc_auc"] = float(model_roc_auc)   # ← AUC as a metric
        mlflow.log_metrics(mlflow_metrics)
        print("Metrics Logged")

        # Log ROC AUC curve image
        mlflow.log_artifact(roc_path, artifact_path="plots/ROC AUC Curve")

        # Log Confusion matrix image
        mlflow.log_artifact(model_cm_path[i], artifact_path="plots/Confusion Matrix")

Initialized MLflow to track repo "JS-Tharun/Customer-Support-Automation"

Repository JS-Tharun/Customer-Support-Automation initialized!

LSTM parameters and tags logged!
LSTM training and validation performance logged!


2026/06/03 13:51:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/03 13:51:27 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


INFO:tensorflow:Assets written to: C:\Users\jstha\AppData\Local\Temp\tmpj7oglde7\model\data\model\assets


INFO:tensorflow:Assets written to: C:\Users\jstha\AppData\Local\Temp\tmpj7oglde7\model\data\model\assets
2026/06/03 13:51:46 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


LSTM model logged!
LSTM metrics logged
Metrics Logged
🏃 View run LSTM at: https://dagshub.com/JS-Tharun/Customer-Support-Automation.mlflow/#/experiments/0/runs/174325f3b07441c79b066e8295f842f9
🧪 View experiment at: https://dagshub.com/JS-Tharun/Customer-Support-Automation.mlflow/#/experiments/0


In [33]:
"""#The first layer is the embedded layer that uses 100 length vectors to represent each word.
#SpatialDropout1D performs variational dropout in NLP models.
#The next layer is the LSTM layer with 100 memory units.
#The output layer must create 13 output values, one for each class.
#Activation function is softmax for multi-class classification.
#Because it is a multi-class classification problem, categorical_crossentropy is used as the loss function.

from keras.models import Sequential
from keras.layers import Embedding,SpatialDropout1D
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential()
model.add(Embedding(MAX_WORDS, EMBEDDING_DIM, input_length=X.shape[1]))
model.add(SpatialDropout1D(0.2))
model.add(LSTM(100, dropout=0.2))
model.add(Dense(7, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

epochs = 20
batch_size = 256

history = model.fit(X_train, Y_train, epochs=epochs, batch_size=batch_size,validation_split=0.1,callbacks=[EarlyStopping(monitor='val_loss', patience=3, min_delta=0.001)])"""

"#The first layer is the embedded layer that uses 100 length vectors to represent each word.\n#SpatialDropout1D performs variational dropout in NLP models.\n#The next layer is the LSTM layer with 100 memory units.\n#The output layer must create 13 output values, one for each class.\n#Activation function is softmax for multi-class classification.\n#Because it is a multi-class classification problem, categorical_crossentropy is used as the loss function.\n\nfrom keras.models import Sequential\nfrom keras.layers import Embedding,SpatialDropout1D\nfrom tensorflow.keras.layers import LSTM\nfrom tensorflow.keras.layers import Dense\nfrom tensorflow.keras.callbacks import EarlyStopping\n\nmodel = Sequential()\nmodel.add(Embedding(MAX_WORDS, EMBEDDING_DIM, input_length=X.shape[1]))\nmodel.add(SpatialDropout1D(0.2))\nmodel.add(LSTM(100, dropout=0.2))\nmodel.add(Dense(7, activation='softmax'))\nmodel.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])\n\nepochs = 20\

In [34]:
"""accr = model.evaluate(X_test,Y_test)
print('Test set\n  Loss: {:0.3f}\n  Accuracy: {:0.3f}'.format(accr[0],accr[1]))"""

"accr = model.evaluate(X_test,Y_test)\nprint('Test set\n  Loss: {:0.3f}\n  Accuracy: {:0.3f}'.format(accr[0],accr[1]))"

In [35]:
"""import matplotlib.pyplot as plt
plt.title('Loss')
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='test')
plt.legend()
plt.show();"""

"import matplotlib.pyplot as plt\nplt.title('Loss')\nplt.plot(history.history['loss'], label='train')\nplt.plot(history.history['val_loss'], label='test')\nplt.legend()\nplt.show();"

In [36]:
"""plt.title('Accuracy')
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='test')
plt.legend()
plt.show();"""

"plt.title('Accuracy')\nplt.plot(history.history['accuracy'], label='train')\nplt.plot(history.history['val_accuracy'], label='test')\nplt.legend()\nplt.show();"

In [37]:
"""#Testing with new custom complaint
import numpy as np

new_complaint = ['I am unable to log in to my account. The system keeps showing a login error even after resetting my password. Please help me access my account.']
seq = tokenizer.texts_to_sequences(new_complaint)
padded = pad_sequences(seq, maxlen=MAX_SEQ_LEN)
pred = model.predict(padded)
labels = pd.get_dummies(df['department']).columns
print(pred, labels[np.argmax(pred)])"""

"#Testing with new custom complaint\nimport numpy as np\n\nnew_complaint = ['I am unable to log in to my account. The system keeps showing a login error even after resetting my password. Please help me access my account.']\nseq = tokenizer.texts_to_sequences(new_complaint)\npadded = pad_sequences(seq, maxlen=MAX_SEQ_LEN)\npred = model.predict(padded)\nlabels = pd.get_dummies(df['department']).columns\nprint(pred, labels[np.argmax(pred)])"

In [38]:
import tensorflow as tf
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]